In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q --upgrade ipython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 626.0/626.0 kB 16.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 12.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.16.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


In [3]:
# Cell 1: Load extension after upgrading
%load_ext autoreload
%autoreload 2

In [4]:
import os
import sys

# Replace with your actual GitHub username and repository name
REPO_URL = "https://github.com/amitkhedar30/adversarial-attack-defense.git"
REPO_NAME = "adversarial-attack-defense"

# Clone if not already present in the Kaggle working directory
if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

# Add repository root to Python path so custom imports work
if f"/kaggle/working/{REPO_NAME}" not in sys.path:
    sys.path.append(f"/kaggle/working/{REPO_NAME}")

print("Repository is ready!")

Cloning into 'adversarial-attack-defense'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 30 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 12.36 KiB | 2.47 MiB/s, done.
Resolving deltas: 100% (6/6), done.
Repository is ready!


In [5]:
# Pull the latest code from GitHub
!cd {REPO_NAME} && git pull

# Install project dependencies
!pip install -q -r {REPO_NAME}/requirements.txt

Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 29.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.3/92.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 12.2 MB/s eta 0:00:00


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

# Import your team's custom modules from the GitHub repository
from models.classifier import get_cifar10_resnet18
from utils.dataset import get_dataloaders
from utils.checkpoint import save_checkpoint, load_latest_checkpoint

# 1. Configuration & Hyperparameters
EPOCHS = 50
BATCH_SIZE = 128
LEARNING_RATE = 0.1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Running on device: {DEVICE}")

# 2. Initialize Data, Model, and Optimizer
train_loader, test_loader = get_dataloaders(batch_size=BATCH_SIZE)
model = get_cifar10_resnet18(num_classes=10).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# 3. Handle Checkpointing (Resumes automatically if Kaggle crashes)
start_epoch = load_latest_checkpoint(model, optimizer, "Standard_ResNet18")

# 4. The Standard Training Loop
print("[*] Starting standard baseline training...")
for epoch in range(start_epoch, EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Track metrics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    scheduler.step()
    
    # Calculate Epoch Metrics
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")
    
    # Save checkpoint at the end of each epoch
    save_checkpoint(model, optimizer, epoch, "Standard_ResNet18")

# 5. Save Final Export Weights
torch.save(model.state_dict(), "/kaggle/working/resnet18_standard_cifar10.pt")
print("[*] Baseline training complete! Final model saved.")

[*] Running on device: cuda


100%|██████████| 170M/170M [20:38<00:00, 138kB/s]  


[*] No existing Standard_ResNet18 checkpoints found. Starting from scratch.
[*] Starting standard baseline training...
Epoch [1/50] | Loss: 1.9499 | Accuracy: 29.88%
[*] Saved Standard_ResNet18 checkpoint to /kaggle/working/checkpoints/Standard_ResNet18_epoch_0.pt
Epoch [2/50] | Loss: 1.4213 | Accuracy: 47.85%
[*] Saved Standard_ResNet18 checkpoint to /kaggle/working/checkpoints/Standard_ResNet18_epoch_1.pt
Epoch [3/50] | Loss: 1.1181 | Accuracy: 59.98%
[*] Saved Standard_ResNet18 checkpoint to /kaggle/working/checkpoints/Standard_ResNet18_epoch_2.pt
Epoch [4/50] | Loss: 0.9212 | Accuracy: 67.45%
[*] Saved Standard_ResNet18 checkpoint to /kaggle/working/checkpoints/Standard_ResNet18_epoch_3.pt
Epoch [5/50] | Loss: 0.7564 | Accuracy: 73.58%
[*] Saved Standard_ResNet18 checkpoint to /kaggle/working/checkpoints/Standard_ResNet18_epoch_4.pt
Epoch [6/50] | Loss: 0.6508 | Accuracy: 77.34%
[*] Saved Standard_ResNet18 checkpoint to /kaggle/working/checkpoints/Standard_ResNet18_epoch_5.pt
Epoch

In [7]:
import torch
from models.classifier import get_cifar10_resnet18
from utils.dataset import get_dataloaders
from attacks.whitebox import fgsm_attack, pgd_attack

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128

# 1. Load the Test Data and Model
_, test_loader = get_dataloaders(batch_size=BATCH_SIZE)
model = get_cifar10_resnet18(num_classes=10)

# Load the weights you just trained!
model.load_state_dict(torch.load("/kaggle/working/resnet18_standard_cifar10.pt"))
model.to(DEVICE)
model.eval() # Crucial: Set model to evaluation mode

# 2. Evaluation Metrics
correct_clean = 0
correct_fgsm = 0
correct_pgd = 0
total = 0

print("[*] Evaluating baseline model against attacks...")
print("This may take a minute or two as it generates attacks...")

for inputs, labels in test_loader:
    inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
    total += labels.size(0)
    
    # A. Clean Accuracy (No Attack)
    with torch.no_grad():
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        correct_clean += predicted.eq(labels).sum().item()
        
    # B. FGSM Attack (epsilon = 8/255 is the standard benchmark)
    inputs_fgsm = fgsm_attack(model, inputs, labels, epsilon=8/255)
    with torch.no_grad():
        outputs_fgsm = model(inputs_fgsm)
        _, pred_fgsm = outputs_fgsm.max(1)
        correct_fgsm += pred_fgsm.eq(labels).sum().item()
        
    # C. PGD Attack (iters=10 is the standard quick-test)
    inputs_pgd = pgd_attack(model, inputs, labels, epsilon=8/255, alpha=2/255, iters=10)
    with torch.no_grad():
        outputs_pgd = model(inputs_pgd)
        _, pred_pgd = outputs_pgd.max(1)
        correct_pgd += pred_pgd.eq(labels).sum().item()

# 3. Print the Devastation
print("-" * 30)
print(f"Clean Test Accuracy: {100. * correct_clean / total:.2f}%")
print(f"FGSM Accuracy:       {100. * correct_fgsm / total:.2f}%")
print(f"PGD Accuracy:        {100. * correct_pgd / total:.2f}%")
print("-" * 30)

[*] Evaluating baseline model against attacks...
This may take a minute or two as it generates attacks...
------------------------------
Clean Test Accuracy: 94.44%
FGSM Accuracy:       33.77%
PGD Accuracy:        0.03%
------------------------------
